# Simuler la variabilité d’une grandeur composée
### Mesurer une distance focale

**Capacité numérique du programme — PCSI et PSI.** Simuler un processus aléatoire pour caractériser la variabilité d’une grandeur composée à l’aide d’un langage de programmation.

**À savoir faire.** Simuler des positions objet et image, calculer une focale pour chaque paire et estimer l’incertitude-type sur la focale.

Pour afficher les commandes de simulation, sélectionner **Run → Run All Cells**.

## Grandeurs et variables Python

| Code | Grandeur | Unité |
|---|---|---|
| <code>p</code> | Position algébrique mesurée de l’objet, $p=\overline{OA}$ | cm |
| <code>p_image</code> | Position algébrique mesurée de l’image, $p'=\overline{OA'}$ | cm |
| <code>u_p</code>, <code>u_image</code> | Incertitudes-types sur les deux positions, $u(p)$ et $u(p')$ | cm |
| <code>n</code> | Nombre de paires simulées | Sans unité |

Les positions mesurées servent de moyennes aux deux lois normales. Les incertitudes-types servent d’écarts-types. Dans ce TP, les deux positions sont supposées indépendantes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from html import escape
from IPython.display import display

try:
    import ipywidgets as widgets
except ModuleNotFoundError:
    import piplite
    await piplite.install("ipywidgets==8.1.8")
    import ipywidgets as widgets

## Lire le calcul Monte-Carlo

Le tableau suit l’ordre de la cellule Python. Les deux appels à <code>np.random.normal</code> sont les deux tirages.

| Code Python | Ce que fait la ligne |
|---|---|
| <code>positions_objet = np.random.normal(p, u_p, n)</code> | Tirer <code>n</code> positions objet. Les trois arguments sont la **moyenne** <code>p</code>, l’**écart-type** <code>u_p</code> et le **nombre de tirages** <code>n</code>. |
| <code>positions_image = np.random.normal(p_image, u_image, n)</code> | Tirer séparément <code>n</code> positions image, cette fois avec la moyenne <code>p_image</code> et l’écart-type <code>u_image</code>. |
| <code>focales = positions_objet * positions_image / (positions_objet - positions_image)</code> | Appliquer la relation de conjugaison $f'=pp'/(p-p')$ à chaque paire. Les deux tableaux de positions donnent un tableau de <code>n</code> focales en cm. |
| <code>u_monte_carlo = np.std(focales, ddof=1)</code> | Calculer l’écart-type des focales simulées. C’est l’estimation Monte-Carlo de l’incertitude-type sur la focale, en cm. |

La fonction ci-dessous regroupe ces quatre lignes et renvoie le tableau des focales et son écart-type.

In [ ]:
def simuler_focales(p, p_image, u_p, u_image, n):
    positions_objet = np.random.normal(p, u_p, n)
    positions_image = np.random.normal(p_image, u_image, n)
    focales = positions_objet * positions_image / (positions_objet - positions_image)
    u_monte_carlo = np.std(focales, ddof=1)
    return focales, u_monte_carlo

## Pour comparer avec le calcul de propagation

Pour des mesures indépendantes et des incertitudes permettant de linéariser la relation de conjugaison,

$$u(f')\simeq
\sqrt{\left(\frac{\partial f'}{\partial p}u(p)\right)^2+
      \left(\frac{\partial f'}{\partial p'}u(p')\right)^2}
=\frac{\sqrt{p'^4u(p)^2+p^4u(p')^2}}{(p-p')^2}.$$

La fonction suivante applique cette formule.

In [ ]:
def propager_incertitudes(p, p_image, u_p, u_image):
    numerateur = np.hypot(p_image**2 * u_p, p**2 * u_image)
    return numerateur / (p - p_image)**2

## Comparer les deux configurations

Pour l’**objet virtuel**, les positions mesurées sont 25 et 20 cm. Pour l’**objet réel**, elles sont −20 et 20 cm. Dans les deux cas, les incertitudes-types valent 0,5 cm.

Choisir un scénario, cliquer sur **Simuler**, puis comparer les histogrammes. L’appel affiché sous les réglages indique les valeurs transmises à la fonction. Chaque clic renouvelle les tirages.

In [ ]:
scenarios = {
    "Objet virtuel (p = +25 cm)": (25, 20, 0.5, 0.5),
    "Objet réel (p = −20 cm)": (-20, 20, 0.5, 0.5),
    "Objet réel, mesures moins précises": (-20, 20, 1.5, 1.5),
}
scenario = widgets.Dropdown(options=list(scenarios), description="Scénario",
                            style={"description_width": "initial"},
                            layout=widgets.Layout(width="min(100%, 420px)"))
positions = [widgets.FloatText(description=nom, layout=widgets.Layout(width="175px"))
             for nom in ("p (cm)", "p′ (cm)", "u(p) (cm)", "u(p′) (cm)")]
tirages = widgets.BoundedIntText(value=1000, min=10, max=100000,
                                 description="Tirages", layout=widgets.Layout(width="175px"))
lancer = widgets.Button(description="Simuler", button_style="primary")
appel = widgets.HTML()
sortie = widgets.Output()
noms_arguments = ("p", "p_image", "u_p", "u_image", "n")
champs = [*positions, tirages]
valeurs_precedentes = None
numero = 0
changement_scenario = False

def actualiser_appel(_=None):
    global valeurs_precedentes
    if changement_scenario:
        return
    valeurs = tuple(champ.value for champ in champs)
    fragments = []
    for i, (nom, valeur) in enumerate(zip(noms_arguments, valeurs)):
        argument = escape(f"{nom}={valeur:g}")
        if valeurs_precedentes is not None and valeur != valeurs_precedentes[i]:
            argument = f"<mark>{argument}</mark>"
        fragments.append(argument)
    appel.value = "<p><b>Appel Monte-Carlo</b><br><code>simuler_focales(" + ", ".join(fragments) + ")</code></p>"
    valeurs_precedentes = valeurs

def choisir_scenario(_=None):
    global changement_scenario
    changement_scenario = True
    try:
        for champ, valeur in zip(positions, scenarios[scenario.value]):
            champ.value = valeur
    finally:
        changement_scenario = False
    actualiser_appel()

def lancer_simulation(_=None):
    global numero
    p, p_image, u_p, u_image, n = (champ.value for champ in champs)
    with sortie:
        sortie.clear_output(wait=True)
        if not np.isfinite([p, p_image, u_p, u_image]).all() or min(u_p, u_image) < 0:
            print("Saisir des positions finies et des incertitudes-types positives ou nulles.")
            return
        if p == p_image:
            print("La relation de conjugaison ne donne pas de focale nominale pour p = p′.")
            return

        focales, u_monte_carlo = simuler_focales(p, p_image, u_p, u_image, n)
        if not np.isfinite(focales).all():
            print("Une paire de positions a donné une focale non finie.")
            return

        ecart = p - p_image
        focale_nominale = p * p_image / ecart
        u_propagation = propager_incertitudes(p, p_image, u_p, u_image)
        numero += 1
        print(f"Simulation {numero} · {n:,} tirages".replace(",", " "))
        print(f"Focale nominale                   {focale_nominale:.4g} cm")
        print(f"Moyenne Monte-Carlo               {np.mean(focales):.4g} cm")
        print(f"Incertitude-type · Monte-Carlo    {u_monte_carlo:.4g} cm")
        print(f"Incertitude-type · propagation    {u_propagation:.4g} cm")
        if abs(ecart) <= 5 * np.hypot(u_p, u_image):
            print("Des paires peuvent approcher p = p′. Examiner la sensibilité de l’écart-type aux tirages.")

        fig, ax = plt.subplots(figsize=(7, 3.5), layout="constrained")
        ax.hist(focales, bins=35, color="#147889", edgecolor="white")
        ax.axvline(focale_nominale, color="#9f3b24", linestyle="--", label="Focale nominale")
        ax.set(xlabel="Distance focale (cm)", ylabel="Nombre de tirages")
        ax.legend()
        plt.show()
        plt.close(fig)

for champ in champs:
    champ.observe(actualiser_appel, names="value")
scenario.observe(choisir_scenario, names="value")
lancer.on_click(lancer_simulation)
choisir_scenario()
display(widgets.VBox([
    scenario,
    widgets.Box(positions, layout=widgets.Layout(display="flex", flex_flow="row wrap")),
    widgets.HBox([tirages, lancer]),
    appel,
    sortie,
]))
lancer_simulation()

## Compléter une ligne

Dans cet extrait de la fonction, les deux premières lignes fournissent les tableaux de positions. Compléter la troisième ligne, avant le calcul de l’écart-type.

```python
positions_objet = np.random.normal(p, u_p, n)
positions_image = np.random.normal(p_image, u_image, n)
focales = …
u_monte_carlo = np.std(focales, ddof=1)
```

Le champ **Essai** sert de brouillon. Sélectionner **Afficher la solution** pour vérifier la ligne, avec ou sans saisie.

In [ ]:
essai = widgets.Text(
    placeholder="Expression après focales =",
    description="Essai",
    layout=widgets.Layout(width="min(100%, 540px)"),
)
solution = widgets.HTML("""
<pre style="white-space: pre-wrap; overflow-wrap: anywhere">focales = positions_objet * positions_image / (positions_objet - positions_image)</pre>
<p>NumPy applique la formule terme à terme. Chaque paire de positions donne une focale.</p>
""")
corrige_ligne = widgets.Accordion(children=[solution], selected_index=None)
corrige_ligne.set_title(0, "Afficher la solution")
display(widgets.VBox([essai, corrige_ligne]))

## Interpréter le résultat

Que mesure l’écart-type des focales simulées ? Pourquoi est-il plus grand dans le scénario « Objet virtuel » ? Formuler une réponse de tête, puis ouvrir l’explication.

In [ ]:
explication = widgets.HTML("""
<p>L’écart-type des focales simulées estime l’incertitude-type sur la focale.
Dans le scénario « Objet virtuel », le dénominateur p − p′ vaut 5 cm,
contre −40 cm pour l’objet réel. À incertitudes de position identiques,
le calcul est plus sensible aux variations des positions quand
la valeur absolue de ce dénominateur est petite.</p>
""")
corrige_interpretation = widgets.Accordion(children=[explication], selected_index=None)
corrige_interpretation.set_title(0, "Afficher l’explication")
display(corrige_interpretation)